# Revision 09a — Fit K=46 leave-one-condition-out spatial MS-AA models

**Recommended repository location**

```text
analysis/04_interpretation/
revision_09a_fit_leave_one_condition_out_K46.ipynb
```

This notebook creates the three targeted fits needed to test whether the
representative shared \(K=46\) spatial MS-AA solution is disproportionately
defined by one experimental condition.

It fits:

1. **drop intact:** word-scrambled + rest;
2. **drop word:** intact + rest;
3. **drop rest:** intact + word-scrambled.

The resulting files are written exactly where
`revision_09_condition_influence_and_seed_stability_K46.ipynb` expects them:

```text
condition_influence_fits/
├── spatialAA_across_drop_intact_K46.npz
├── spatialAA_across_drop_word_K46.npz
└── spatialAA_across_drop_rest_K46.npz
```

## Scientific design

- The fit orientation is spatial MS-AA.
- Model order is fixed at \(K=46\).
- Each remaining condition is preprocessed exactly as in the canonical
  three-condition workflow: matrices are oriented time-by-node, rest is
  truncated from 400 to 300 TRs, and each condition is normalized separately
  using the project HyperTools `normalize(..., normalize="across")` function.
- The normalized condition blocks are then concatenated in canonical order.
- Fitting options and random seed are held fixed across omission fits.
- The notebook is resumable: valid existing files are skipped unless
  `FORCE_REFIT=True`.

A condition-omission effect should be interpreted only after aligning the
resulting archetypes to the canonical full-condition fit. That alignment is
performed by the companion Revision 09 notebook, not here.

### Revision 09a.1 dependency fix

The project HyperTools normalizer is loaded with its containing directory
temporarily exposed on `sys.path`, allowing local imports such as
`format_data` and `helpers` to resolve. Because inputs are already numeric
time-by-node arrays, normalization is called with `format_data=False`.


In [ ]:
# ============================================================
# USER SETTINGS
# ============================================================

from pathlib import Path

ANALYSIS_TYPE = "spatial"
K = 46
ALL_CONDITIONS = ["intact", "word", "rest"]

# Run all three, or set a subset while debugging / submitting jobs.
FITS_TO_RUN = [
    "drop_intact",
    "drop_word",
    "drop_rest",
]

# Canonical project paths are resolved from the repository root.
PIEMAN_MAT_CANDIDATES = [
    Path("data/pieman/raw/pieman_data.mat"),
    Path("data/input/pieman_data.mat"),
    Path("pieman_data.mat"),
]

POSTERIOR_MAT_CANDIDATES = [
    Path("data/pieman/raw/pieman_posterior_K700.mat"),
    Path("data/input/pieman_posterior_K700.mat"),
    Path("pieman_posterior_K700.mat"),
]

CANONICAL_FIT = Path(
    "msaa_flexible_outputs_npz/"
    "spatialAA_across_acrossCond_K46.npz"
)

OUTPUT_DIR = Path("condition_influence_fits")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Match the original primary analysis.
TRIM_REST_BY = 100
RNG_SEED = 42

MSAA_OPTS = {
    "maxiter": 200,
    "conv_crit": 1e-6,
    "fix_var_iter": 5,
    "heteroscedastic": False,
    "use_gpu": False,
    "rngSEED": RNG_SEED,
}

# Safe restart behavior.
SKIP_EXISTING = True
FORCE_REFIT = False
STOP_ON_ERROR = True

# A silent fallback could make the omission fits differ from the canonical fit.
# Keep False for manuscript analyses. Turn True only for diagnostic testing.
ALLOW_MANUAL_NORMALIZATION_FALLBACK = False

# Optional control: repeat the full intact+word+rest K=46 fit with this exact
# notebook and compare it to CANONICAL_FIT. This costs one additional fit.
RUN_FULL_CONDITION_CONTROL = True

OUTPUT_FILES = {
    "drop_intact": (
        OUTPUT_DIR
        / "spatialAA_across_drop_intact_K46.npz"
    ),
    "drop_word": (
        OUTPUT_DIR
        / "spatialAA_across_drop_word_K46.npz"
    ),
    "drop_rest": (
        OUTPUT_DIR
        / "spatialAA_across_drop_rest_K46.npz"
    ),
    "full_control": (
        OUTPUT_DIR
        / "spatialAA_across_full_control_K46.npz"
    ),
}

INCLUDED_CONDITIONS = {
    "drop_intact": ["word", "rest"],
    "drop_word": ["intact", "rest"],
    "drop_rest": ["intact", "word"],
    "full_control": ["intact", "word", "rest"],
}

print("K:", K)
print("Fitting seed:", RNG_SEED)
print("Jobs:", FITS_TO_RUN)
print("Output:", OUTPUT_DIR.resolve())


In [ ]:
# ============================================================
# IMPORTS AND REPOSITORY-LOCAL DEPENDENCIES
# ============================================================

import hashlib
import importlib.util
import inspect
import json
import os
import sys
import time
import traceback
import warnings

import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.optimize import linear_sum_assignment
from IPython.display import display


REPO = Path.cwd().resolve()

# The notebook should be executed from the repository root by
# scripts/execute_notebook.py. Fail early if that assumption is violated.
if not (REPO / "analysis").exists():
    warnings.warn(
        "The current working directory does not look like the repository root. "
        "Run with scripts/execute_notebook.py from the repo root."
    )


def first_existing(candidates, label):
    for path in candidates:
        path = Path(path)
        if path.exists():
            print(f"{label}: {path}")
            return path

    raise FileNotFoundError(
        f"Could not find {label}. Tried:\n"
        + "\n".join(f"  - {Path(p)}" for p in candidates)
    )


PIEMAN_MAT_PATH = first_existing(
    PIEMAN_MAT_CANDIDATES,
    "Pieman data",
)

POSTERIOR_MAT_PATH = first_existing(
    POSTERIOR_MAT_CANDIDATES,
    "HTFA posterior",
)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# Load the exact repository MS-AA implementation.
# ------------------------------------------------------------

helper_candidates = [
    REPO / "helpers",
    REPO,
    REPO / "src",
]

for candidate in helper_candidates:
    if candidate.exists():
        sys.path.insert(0, str(candidate))

try:
    from MultiSubject_AA import Subject, multi_subject_aa
except Exception as exc:
    helper_hits = [
        path
        for path in REPO.glob("**/MultiSubject_AA.py")
        if ".git" not in path.parts
        and "__pycache__" not in path.parts
    ]

    if not helper_hits:
        raise ImportError(
            "Could not import MultiSubject_AA and could not locate "
            "MultiSubject_AA.py in the repository."
        ) from exc

    # Prefer the curated helpers directory.
    helper_hits = sorted(
        helper_hits,
        key=lambda p: (
            0 if "helpers" in p.parts else 1,
            len(p.parts),
            str(p),
        ),
    )

    helper_dir = helper_hits[0].parent
    sys.path.insert(0, str(helper_dir))

    from MultiSubject_AA import Subject, multi_subject_aa

print("MS-AA implementation:", inspect.getfile(multi_subject_aa))


# ------------------------------------------------------------
# Load the project HyperTools normalization function.
#
# The historical hypertools/normalize.py uses local imports such as
# `from format_data import ...` and `from helpers import ...`. Loading the
# file directly does not automatically expose its containing directory, so
# we temporarily add that directory to sys.path while executing the module.
# ------------------------------------------------------------

NORMALIZER_SOURCE = None
normalize_across = None


def load_project_normalize_module(path):
    path = Path(path).expanduser().resolve()

    if not path.exists():
        raise FileNotFoundError(
            f"Project normalize.py not found: {path}"
        )

    module_dir = str(path.parent)
    added_path = module_dir not in sys.path

    if added_path:
        sys.path.insert(0, module_dir)

    try:
        spec = importlib.util.spec_from_file_location(
            "project_hypertools_normalize",
            str(path),
        )

        if spec is None or spec.loader is None:
            raise ImportError(
                f"Could not create an import spec for {path}"
            )

        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)

    finally:
        if added_path:
            try:
                sys.path.remove(module_dir)
            except ValueError:
                pass

    return module


normalizer_candidates = [
    REPO / "hypertools" / "normalize.py",
    REPO / "helpers" / "hypertools" / "normalize.py",
]

normalizer_import_errors = []

for path in normalizer_candidates:
    if not path.exists():
        continue

    try:
        module = load_project_normalize_module(path)
    except Exception as exc:
        normalizer_import_errors.append(
            f"{path}: {type(exc).__name__}: {exc}"
        )
        continue

    if hasattr(module, "normalize"):
        normalize_across = module.normalize
        NORMALIZER_SOURCE = str(path.resolve())
        break

if normalize_across is None:
    try:
        from hypertools.tools.normalize import normalize as normalize_across
        NORMALIZER_SOURCE = (
            "installed hypertools.tools.normalize.normalize"
        )
    except Exception as exc:
        normalizer_import_errors.append(
            "installed hypertools: "
            f"{type(exc).__name__}: {exc}"
        )
        normalize_across = None

print("Normalizer source:", NORMALIZER_SOURCE)

if normalize_across is None and normalizer_import_errors:
    print("Normalizer import attempts:")
    for message in normalizer_import_errors:
        print(" -", message)


## 1. Load and reproduce the canonical condition-wise preprocessing

The canonical workflow normalizes each condition separately before concatenating
condition blocks for the across-condition fit. Therefore, dropping a condition
does not alter the normalization of the remaining condition blocks.


In [ ]:
# ============================================================
# DATA EXTRACTION / ORIENTATION / NORMALIZATION
# ============================================================

def to_float_array(x):
    return np.asarray(x, dtype=float)


def unwrap_matlab_cell(value):
    current = value

    # MATLAB cells sometimes arrive with several singleton object layers.
    while (
        isinstance(current, np.ndarray)
        and current.dtype == object
        and current.size == 1
    ):
        current = current.reshape(-1)[0]

    return np.asarray(current, dtype=float)


def extract_condition_cells(mat, condition):
    if condition not in mat:
        raise KeyError(
            f"Condition {condition!r} not found. "
            f"Available non-metadata keys: "
            f"{[k for k in mat if not k.startswith('__')]}"
        )

    arr = np.asarray(mat[condition], dtype=object)

    if arr.ndim != 2:
        raise ValueError(
            f"Expected a 2D MATLAB cell array for {condition}; "
            f"found {arr.shape}."
        )

    # Preserve the column order used by the original notebooks.
    if arr.shape[0] == 1:
        cells = [arr[0, i] for i in range(arr.shape[1])]
    elif arr.shape[1] == 1:
        cells = [arr[i, 0] for i in range(arr.shape[0])]
    else:
        cells = list(arr.ravel(order="F"))

    return [unwrap_matlab_cell(cell) for cell in cells]


def coerce_TxV(x):
    x = to_float_array(x)

    if x.ndim != 2:
        raise ValueError(
            f"Expected a 2D subject matrix, found {x.shape}."
        )

    T, V = x.shape

    # HTFA has 700 nodes and 300/400 timepoints.
    if V < T:
        x = x.T
        T, V = x.shape

    # Match the canonical wrapper: remove each node's temporal mean.
    x = x - x.mean(axis=0, keepdims=True)

    return x.astype(np.float32, copy=False)


def manual_normalize_across(subject_list, eps=1e-8):
    subject_list = [coerce_TxV(x) for x in subject_list]
    stacked = np.vstack(subject_list)

    mu = stacked.mean(axis=0, keepdims=True)
    sd = stacked.std(axis=0, keepdims=True) + eps

    return [
        ((x - mu) / sd).astype(np.float32, copy=False)
        for x in subject_list
    ]


def normalize_condition(subject_list, condition):
    oriented = [coerce_TxV(x) for x in subject_list]

    if condition == "rest" and TRIM_REST_BY > 0:
        oriented = [
            x[:-TRIM_REST_BY].copy()
            for x in oriented
        ]

    if normalize_across is not None:
        # Inputs have already been converted to numeric T x V arrays.
        # Bypass HyperTools' legacy formatting layer; this keeps the exact
        # across-normalization logic while avoiding unrelated optional
        # dependencies (df2mat, text2mat, datageometry, etc.).
        normalized = normalize_across(
            oriented,
            normalize="across",
            format_data=False,
        )
        normalized = [
            np.asarray(x, dtype=np.float32)
            for x in normalized
        ]
        source = NORMALIZER_SOURCE
    elif ALLOW_MANUAL_NORMALIZATION_FALLBACK:
        warnings.warn(
            "Using manual across-normalization fallback. "
            "Do not use fallback-generated fits for the final paper "
            "unless equivalence to the canonical normalizer is verified."
        )
        normalized = manual_normalize_across(oriented)
        source = "manual fallback"
    else:
        raise RuntimeError(
            "The canonical HyperTools normalizer was not found. "
            "Restore hypertools/normalize.py or install the matching "
            "HyperTools version. The manuscript notebook intentionally "
            "refuses to fall back silently."
        )

    shapes = {
        tuple(np.asarray(x).shape)
        for x in normalized
    }

    if len(shapes) != 1:
        raise ValueError(
            f"{condition}: inconsistent normalized shapes: {shapes}"
        )

    return normalized, source


pieman_mat = loadmat(PIEMAN_MAT_PATH)

raw_condition_data = {
    condition: extract_condition_cells(
        pieman_mat,
        condition,
    )
    for condition in ALL_CONDITIONS
}

condition_data = {}
normalizer_sources = {}

for condition in ALL_CONDITIONS:
    condition_data[condition], normalizer_sources[condition] = (
        normalize_condition(
            raw_condition_data[condition],
            condition,
        )
    )

condition_qc_rows = []

for condition in ALL_CONDITIONS:
    arrays = condition_data[condition]
    stacked = np.vstack(arrays)

    condition_qc_rows.append(
        {
            "condition": condition,
            "n_observations": len(arrays),
            "T": int(arrays[0].shape[0]),
            "V": int(arrays[0].shape[1]),
            "grand_mean": float(stacked.mean()),
            "grand_sd": float(stacked.std()),
            "normalizer": normalizer_sources[condition],
        }
    )

condition_qc_df = pd.DataFrame(condition_qc_rows)
display(condition_qc_df)

if not all(
    tuple(condition_data[c][0].shape) == (300, 700)
    for c in ALL_CONDITIONS
):
    raise ValueError(
        "Expected every primary condition to have shape (300,700) "
        "after orientation and rest trimming."
    )

if not all(
    len(condition_data[c]) == 36
    for c in ALL_CONDITIONS
):
    warnings.warn(
        "Expected 36 observations per primary condition. "
        "Review the displayed counts before fitting."
    )

condition_qc_df.to_csv(
    OUTPUT_DIR / "leave_one_condition_out_input_qc.csv",
    index=False,
)


In [ ]:
# ============================================================
# LOAD HTFA GEOMETRY AND CANONICAL FIT METADATA
# ============================================================

posterior = loadmat(POSTERIOR_MAT_PATH)

centers = np.asarray(
    posterior["posterior"]["centers"][0][0][0][0][0],
    dtype=float,
)

widths = np.asarray(
    posterior["posterior"]["widths"][0][0][0][0][0][:, 0],
    dtype=float,
).ravel()

print("centers:", centers.shape)
print("widths:", widths.shape)

if centers.shape != (700, 3):
    warnings.warn(
        f"Expected centers shape (700,3), found {centers.shape}."
    )

canonical_metadata = {}

if CANONICAL_FIT.exists():
    canonical_npz = np.load(
        CANONICAL_FIT,
        allow_pickle=True,
    )

    canonical_metadata = {
        "K": int(np.asarray(canonical_npz["K"]).item()),
        "n_entries": len(
            canonical_npz["results_subj"].tolist()
        ),
        "keys": sorted(canonical_npz.files),
    }

    if "C" in canonical_npz:
        canonical_metadata["C_shape"] = tuple(
            np.asarray(
                canonical_npz["C"],
                dtype=float,
            ).shape
        )

    print("Canonical fit metadata:")
    print(json.dumps(canonical_metadata, indent=2))
else:
    warnings.warn(
        f"Canonical fit not found at {CANONICAL_FIT}. "
        "The omission fits can still run, but the optional full-control "
        "comparison will be unavailable."
    )


## 2. Fit and save functions

Each saved NPZ preserves the existing project schema and adds explicit
provenance fields identifying the omitted condition, included conditions,
input hashes, normalization method, fit options, and source implementation.


In [ ]:
# ============================================================
# FIT / SAVE / VALIDATION HELPERS
# ============================================================

GLOBAL_CONDITION_CODE = {
    "intact": 0,
    "word": 1,
    "rest": 2,
}


def build_subject_objects(subject_arrays):
    return [
        Subject(
            X=np.asarray(x, dtype=np.float32),
            sX=np.asarray(x, dtype=np.float32),
        )
        for x in subject_arrays
    ]


def build_job_data(included_conditions):
    arrays = []
    labels = []

    for condition in ALL_CONDITIONS:
        if condition not in included_conditions:
            continue

        arrays.extend(condition_data[condition])
        labels.extend(
            [condition] * len(condition_data[condition])
        )

    return arrays, np.asarray(labels, dtype=object)


def atomic_save_fit(
    save_path,
    *,
    results_subj,
    C,
    cost_fun,
    varexpl,
    secs,
    condition_labels,
    included_conditions,
    dropped_condition,
    job_name,
):
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    condition_codes = np.asarray(
        [
            GLOBAL_CONDITION_CODE[str(label)]
            for label in condition_labels
        ],
        dtype=int,
    )

    tmp_path = save_path.with_suffix(".tmp.npz")

    np.savez_compressed(
        tmp_path,
        K=np.asarray(K),
        results_subj=np.asarray(
            results_subj,
            dtype=object,
        ),
        C=np.asarray(C, dtype=float),
        cost_fun=np.asarray(cost_fun, dtype=float),
        varexpl=np.asarray(varexpl, dtype=float),
        secs=np.asarray(secs, dtype=float),
        condition_labels_str=np.asarray(
            condition_labels,
            dtype=object,
        ),
        condition_codes=condition_codes,
        condition_names=np.asarray(
            included_conditions,
            dtype=object,
        ),
        analysis_type=np.asarray(
            ANALYSIS_TYPE,
            dtype=object,
        ),
        fit_scope=np.asarray(
            "across",
            dtype=object,
        ),
        dropped_condition=np.asarray(
            dropped_condition,
            dtype=object,
        ),
        included_conditions=np.asarray(
            included_conditions,
            dtype=object,
        ),
        job_name=np.asarray(
            job_name,
            dtype=object,
        ),
        rng_seed=np.asarray(
            RNG_SEED,
            dtype=int,
        ),
        fit_options_json=np.asarray(
            json.dumps(MSAA_OPTS, sort_keys=True),
            dtype=object,
        ),
        preprocessing_json=np.asarray(
            json.dumps(
                {
                    "condition_wise_normalization": True,
                    "normalization": (
                        "hypertools normalize='across'"
                    ),
                    "normalizer_sources": normalizer_sources,
                    "rest_trim_last_TRs": TRIM_REST_BY,
                    "orientation": "time_by_node",
                    "temporal_mean_removed_per_node": True,
                },
                sort_keys=True,
            ),
            dtype=object,
        ),
        pieman_data_sha256=np.asarray(
            sha256_file(PIEMAN_MAT_PATH),
            dtype=object,
        ),
        posterior_sha256=np.asarray(
            sha256_file(POSTERIOR_MAT_PATH),
            dtype=object,
        ),
        implementation_file=np.asarray(
            inspect.getfile(multi_subject_aa),
            dtype=object,
        ),
        centers=np.asarray(centers, dtype=float),
        widths=np.asarray(widths, dtype=float),
    )

    tmp_path.replace(save_path)


def validate_saved_fit(path, expected_conditions):
    path = Path(path)

    data = np.load(path, allow_pickle=True)
    results_subj = data["results_subj"].tolist()
    labels = np.asarray(
        data["condition_labels_str"]
    ).astype(str)
    C = np.asarray(data["C"], dtype=float)
    cost = np.asarray(data["cost_fun"], dtype=float)

    expected_n = sum(
        len(condition_data[c])
        for c in expected_conditions
    )

    checks = {
        "path": str(path),
        "K": int(np.asarray(data["K"]).item()),
        "n_entries": len(results_subj),
        "expected_n_entries": expected_n,
        "C_shape": tuple(C.shape),
        "sXC_shape": tuple(
            np.asarray(results_subj[0]["sXC"]).shape
        ),
        "S_shape": tuple(
            np.asarray(results_subj[0]["S"]).shape
        ),
        "column_sum_max_error": float(
            np.max(
                np.abs(
                    C.sum(axis=0) - 1.0
                )
            )
        ),
        "C_min": float(np.min(C)),
        "cost_iterations": len(cost),
        "cost_initial": float(cost[0]) if len(cost) else np.nan,
        "cost_final": float(cost[-1]) if len(cost) else np.nan,
        "varexpl": float(np.asarray(data["varexpl"]).item()),
        "secs": float(np.asarray(data["secs"]).item()),
    }

    if checks["K"] != K:
        raise ValueError(checks)

    if checks["n_entries"] != expected_n:
        raise ValueError(checks)

    if checks["C_shape"] != (700, K):
        raise ValueError(checks)

    if checks["sXC_shape"] != (300, K):
        raise ValueError(checks)

    if checks["S_shape"] != (K, 700):
        raise ValueError(checks)

    if set(labels) != set(expected_conditions):
        raise ValueError(
            f"{path}: labels {set(labels)} != "
            f"{set(expected_conditions)}"
        )

    if not np.isfinite(cost).all():
        raise ValueError(
            f"{path}: non-finite optimization cost."
        )

    return checks


def existing_fit_is_valid(path, expected_conditions):
    try:
        validate_saved_fit(
            path,
            expected_conditions,
        )
        return True
    except Exception:
        return False


## 3. Run the three omission fits

This cell checkpoints after every completed fit. You can run one job at a time
by editing `FITS_TO_RUN`; already validated outputs will be reused.


In [ ]:
# ============================================================
# RUN / RESUME FITS
# ============================================================

fit_manifest_rows = []

jobs = list(FITS_TO_RUN)

if RUN_FULL_CONDITION_CONTROL:
    jobs = ["full_control"] + jobs

for job_name in jobs:
    if job_name not in INCLUDED_CONDITIONS:
        raise KeyError(
            f"Unknown job {job_name!r}."
        )

    included_conditions = INCLUDED_CONDITIONS[job_name]
    dropped_condition = (
        None
        if job_name == "full_control"
        else job_name.replace("drop_", "")
    )
    save_path = OUTPUT_FILES[job_name]

    print("\n" + "=" * 88)
    print("JOB:", job_name)
    print("Included:", included_conditions)
    print("Dropped:", dropped_condition)
    print("Save:", save_path)
    print("=" * 88, flush=True)

    if (
        save_path.exists()
        and SKIP_EXISTING
        and not FORCE_REFIT
        and existing_fit_is_valid(
            save_path,
            included_conditions,
        )
    ):
        print("Validated existing fit; skipping.")

        qc = validate_saved_fit(
            save_path,
            included_conditions,
        )

        fit_manifest_rows.append(
            {
                "job_name": job_name,
                "status": "reused_valid_existing",
                "dropped_condition": dropped_condition,
                "included_conditions": "|".join(
                    included_conditions
                ),
                **qc,
            }
        )
        continue

    try:
        subject_arrays, condition_labels = (
            build_job_data(included_conditions)
        )

        print(
            "N observations:",
            len(subject_arrays),
        )
        print(
            "Condition counts:",
            pd.Series(condition_labels).value_counts().to_dict(),
        )
        print(
            "Example shape:",
            np.asarray(subject_arrays[0]).shape,
        )
        print(
            "Fit options:",
            json.dumps(MSAA_OPTS, indent=2),
        )

        subject_objects = build_subject_objects(
            subject_arrays
        )

        start = time.perf_counter()

        results_subj, C, cost_fun, varexpl, secs = (
            multi_subject_aa(
                subject_objects,
                noc=K,
                opts=dict(MSAA_OPTS),
            )
        )

        wall_secs = time.perf_counter() - start

        atomic_save_fit(
            save_path,
            results_subj=results_subj,
            C=C,
            cost_fun=cost_fun,
            varexpl=varexpl,
            secs=secs,
            condition_labels=condition_labels,
            included_conditions=included_conditions,
            dropped_condition=dropped_condition,
            job_name=job_name,
        )

        qc = validate_saved_fit(
            save_path,
            included_conditions,
        )

        print(
            "Finished:",
            job_name,
            "| variance explained:",
            f"{100 * qc['varexpl']:.2f}%",
            "| helper secs:",
            f"{qc['secs']:.1f}",
            "| wall secs:",
            f"{wall_secs:.1f}",
        )
        print(
            "Cost:",
            qc["cost_initial"],
            "->",
            qc["cost_final"],
            "| iterations:",
            qc["cost_iterations"],
        )

        fit_manifest_rows.append(
            {
                "job_name": job_name,
                "status": "fit_complete",
                "dropped_condition": dropped_condition,
                "included_conditions": "|".join(
                    included_conditions
                ),
                "wall_secs": wall_secs,
                **qc,
            }
        )

        # Release the large in-memory structures before the next fit.
        del subject_objects
        del results_subj
        del C
        del cost_fun

    except Exception as exc:
        traceback.print_exc()

        fit_manifest_rows.append(
            {
                "job_name": job_name,
                "status": "failed",
                "dropped_condition": dropped_condition,
                "included_conditions": "|".join(
                    included_conditions
                ),
                "path": str(save_path),
                "error": repr(exc),
            }
        )

        if STOP_ON_ERROR:
            raise


fit_manifest_df = pd.DataFrame(
    fit_manifest_rows
)

display(fit_manifest_df)

manifest_path = (
    OUTPUT_DIR
    / "leave_one_condition_out_fit_manifest.csv"
)

fit_manifest_df.to_csv(
    manifest_path,
    index=False,
)

print("Saved manifest:", manifest_path)


## 4. Optional full-condition control alignment

When `RUN_FULL_CONDITION_CONTROL=True`, this cell aligns the newly generated
full-condition control fit to the saved canonical \(K=46\) fit. Near-perfect
agreement confirms that preprocessing, implementation, seed, and fitting
settings match the canonical analysis. If agreement is not near-perfect, do
not attribute leave-one-condition-out differences solely to the omitted
condition until the pipeline mismatch is resolved.


In [ ]:
# ============================================================
# OPTIONAL CANONICAL CONTROL CHECK
# ============================================================

def safe_column_cosine(A, B):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)

    A = A / (
        np.linalg.norm(A, axis=0, keepdims=True)
        + 1e-12
    )
    B = B / (
        np.linalg.norm(B, axis=0, keepdims=True)
        + 1e-12
    )

    return A.T @ B


full_control_path = OUTPUT_FILES["full_control"]

if (
    RUN_FULL_CONDITION_CONTROL
    and CANONICAL_FIT.exists()
    and full_control_path.exists()
):
    canonical_npz = np.load(
        CANONICAL_FIT,
        allow_pickle=True,
    )
    control_npz = np.load(
        full_control_path,
        allow_pickle=True,
    )

    canonical_C = np.asarray(
        canonical_npz["C"],
        dtype=float,
    )
    control_C = np.asarray(
        control_npz["C"],
        dtype=float,
    )

    similarity = safe_column_cosine(
        canonical_C,
        control_C,
    )

    ref_idx, control_idx = linear_sum_assignment(
        -similarity
    )

    matched = similarity[
        ref_idx,
        control_idx,
    ]

    control_alignment_df = pd.DataFrame(
        {
            "canonical_archetype": ref_idx,
            "control_archetype": control_idx,
            "generator_cosine": matched,
        }
    )

    display(control_alignment_df.head(12))

    summary = {
        "mean": float(np.mean(matched)),
        "median": float(np.median(matched)),
        "minimum": float(np.min(matched)),
        "n_ge_0.99": int(np.sum(matched >= 0.99)),
        "n_archetypes": int(len(matched)),
    }

    print("Full-control alignment:")
    print(json.dumps(summary, indent=2))

    control_alignment_df.to_csv(
        OUTPUT_DIR
        / "full_control_to_canonical_alignment.csv",
        index=False,
    )

    with open(
        OUTPUT_DIR
        / "full_control_to_canonical_summary.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(summary, handle, indent=2)

    if summary["minimum"] < 0.95:
        warnings.warn(
            "The full-control fit does not closely reproduce the "
            "canonical fit. Check seed, implementation, preprocessing, "
            "normalization, and stopping settings before interpreting "
            "condition-omission differences."
        )
else:
    print(
        "Full-condition control comparison skipped. "
        "Set RUN_FULL_CONDITION_CONTROL=True to run it."
    )


## 5. Run the condition-influence analysis

After all three omission fits complete, execute:

```bash
python scripts/execute_notebook.py \
  analysis/04_interpretation/revision_09_condition_influence_and_seed_stability_K46.ipynb
```

That companion notebook will:

- align each omission fit to the canonical full-condition solution;
- compare generator-column cosine similarity;
- compare temporal motifs at zero lag and within \(\pm2\) TRs;
- compare spatial deployment maps;
- summarize the five held-out-selected archetypes separately.

## Recommended run order

```bash
python scripts/execute_notebook.py \
  analysis/04_interpretation/revision_09a_fit_leave_one_condition_out_K46.ipynb

python scripts/execute_notebook.py \
  analysis/04_interpretation/revision_09_condition_influence_and_seed_stability_K46.ipynb
```
